## Thêm dữ liệu vào MYSQL

In [ ]:
import pandas as pd
import mysql.connector

# --- 1. Cấu hình kết nối MySQL ---
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="12345",
    database="olist"
)
cursor = conn.cursor()

# --- 2. Hàm chèn CSV vào MySQL theo batch ---
def insert_csv_to_mysql(csv_file, table_name, rename_columns=None, batch_size=8000):
    df = pd.read_csv(csv_file)
    
    # --- Đổi tên cột nếu cần ---
    if rename_columns:
        df.rename(columns=rename_columns, inplace=True)
    
    columns = ", ".join(df.columns)
    placeholders = ", ".join(["%s"] * len(df.columns))
    sql = f"INSERT INTO {table_name} ({columns}) VALUES ({placeholders})"
    
    # --- Convert NaN → None để MySQL hiểu là NULL ---
    data = [
        tuple(None if (isinstance(x, float) and pd.isna(x)) else x for x in row)
        for row in df.to_numpy()
    ]
    
    # --- Chia batch và insert ---
    for i in range(0, len(data), batch_size):
        batch = data[i:i+batch_size]
        cursor.executemany(sql, batch)
        conn.commit()
    

# --- 3. Ví dụ insert orders ---
# insert_csv_to_mysql("../../data/2_clean/customers.csv", "customers")
# geolocation_map = {"long": "longtitude", "lat": "latitude"}
# insert_csv_to_mysql("../../data/2_clean/geolocation.csv", "geolocation", rename_columns=geolocation_map)
# insert_csv_to_mysql("../../data/2_clean/order_items.csv", "order_items")
# insert_csv_to_mysql("../../data/2_clean/payments.csv", "payments")
# insert_csv_to_mysql("../../data/2_clean/sellers.csv", "sellers")
# insert_csv_to_mysql("../../data/2_clean/reviews.csv", "reviews")
# insert_csv_to_mysql("../../data/2_clean/products.csv", "products")
# insert_csv_to_mysql("../../data/2_clean/orders.csv", "orders")
insert_csv_to_mysql("../../data/2_clean/orders.csv", "orders")

# --- 4. Đóng kết nối ---
cursor.close()
conn.close()